# Problem: Implement an LSTM Model 

### Problem Statement
You are tasked with implementing a simple **LSTM (Long Short-Term Memory)** model in PyTorch. The model should process sequential data using an LSTM layer followed by a fully connected (FC) layer. Your goal is two-fold: one is to implement a LSTM layer from scratch and another using inbuilt pytorch LSTM layer. Compare the results implementing the forward passes for both the LSTM models.

### Requirements
1. **Define the LSTM Model using a Custom LSTM layer**:
   - Add a `Custom` LSTM layer to the model. The layer must take care of the hidden and cell states
   - Add a **fully connected (FC) layer** that maps the output of the LSTM to the final predictions.
   - Implement the `forward` method to:
     - Pass the input sequence through the LSTM.
     - Feed the output of the LSTM into the fully connected layer for the final output.

2. **Define the LSTM Model using an in-built LSTM layer**:
  - Same as `1` with the only difference that this time the LSTM layer is PyTorch's `nn.LSTM`

3. **Check the custom layer against `nn.LSTM`**: copy your weights into an `nn.LSTM` and confirm both produce the same hidden states. `nn.LSTM` stacks the four gates in the order input, forget, cell candidate, output (`weight_ih_l0` is `(4*hidden, input)`), and adds two bias vectors, `bias_ih_l0 + bias_hh_l0`.

### Interface of the custom model
- `CustomLSTMModel(input_dim, hidden_units).forward(inputs, H_C=None)` takes `inputs` of shape `(batch, seq_len, input_dim)` and returns `(predictions, (H, C))`: one prediction per timestep, `(batch, seq_len, 1)`, plus the final hidden and cell states, each `(batch, hidden_units)`.
- When `H_C` is `None`, start from zero hidden and cell states (as `nn.LSTM` does); otherwise continue from the given `(H, C)`.

### Constraints
- The LSTM layer should be implemented with a single hidden layer.
- Use a suitable number of input features, hidden units, and output size for the task.
- Make sure the `forward` method returns the output of the fully connected layer after processing the LSTM output.


<details>
  <summary>💡 Hint</summary>
  Add the LSTM layer and FC layer in LSTMModel.__init__.
  <br>
  Implement the forward pass to process sequences using the LSTM and FC layers.
  <br> Review Hidden and cell states computation here: [D2l.ai](https://d2l.ai/chapter_recurrent-modern/lstm.html)
</details>

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
# Generate synthetic sequential data
torch.manual_seed(42)
sequence_length = 10
num_samples = 100

# Create a sine wave dataset
X = torch.linspace(0, 4 * 3.14159, steps=num_samples).unsqueeze(1)
y = torch.sin(X)

# Prepare data for LSTM
def create_in_out_sequences(data, seq_length):
    in_seq = []
    out_seq = []
    for i in range(len(data) - seq_length):
        in_seq.append(data[i:i + seq_length])
        out_seq.append(data[i + seq_length])
    return torch.stack(in_seq), torch.stack(out_seq)

X_seq, y_seq = create_in_out_sequences(y, sequence_length)

In [ ]:
class CustomLSTMModel(nn.Module):
    """Single-layer LSTM written out by hand, followed by a Linear head.

    The parameters use nn.LSTM's layout and gate order (input, forget, cell
    candidate, output), so the weights can be copied into nn.LSTM to check the math.
    """

    def __init__(self, input_dim, hidden_units):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_units = hidden_units
        H = hidden_units
        # The four gates are stacked so that each step needs one matmul per input instead of
        # four: rows [0:H] input gate, [H:2H] forget gate, [2H:3H] candidate cell, [3H:4H] output gate.
        self.W_ih = nn.Parameter(torch.empty(4 * H, input_dim))
        self.W_hh = nn.Parameter(torch.empty(4 * H, H))
        self.bias = nn.Parameter(torch.empty(4 * H))
        # nn.LSTM's init, U(-1/sqrt(H), 1/sqrt(H)). Unit-variance weights would give gate
        # pre-activations with std ~sqrt(H), saturating the sigmoids/tanhs and killing the gradients.
        bound = H ** -0.5
        for p in (self.W_ih, self.W_hh, self.bias):
            nn.init.uniform_(p, -bound, bound)
        self.fc = nn.Linear(H, 1)

    def forward(self, inputs, H_C=None):
        """inputs: (batch, seq_len, input_dim) -> (predictions (batch, seq_len, 1), (H, C))."""
        batch_size, seq_len, _ = inputs.shape
        if H_C is None:
            # Zero initial state, like nn.LSTM. (A random one would make the output depend
            # on the call rather than only on the input.)
            H = inputs.new_zeros(batch_size, self.hidden_units)
            C = inputs.new_zeros(batch_size, self.hidden_units)
        else:
            H, C = H_C

        # The input projection does not depend on the recurrence: compute it for all timesteps at once
        x_proj = inputs @ self.W_ih.T + self.bias              # (batch, seq_len, 4H)

        all_hidden_states = []
        for t in range(seq_len):
            gates = x_proj[:, t] + H @ self.W_hh.T              # (batch, 4H)
            i, f, g, o = gates.chunk(4, dim=1)
            I_t, F_t, O_t = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
            C_tilde = torch.tanh(g)
            C = F_t * C + I_t * C_tilde                         # forget part of the memory, write new content
            H = O_t * torch.tanh(C)                             # expose a gated view of the memory
            all_hidden_states.append(H)

        outputs = torch.stack(all_hidden_states, dim=1)         # (batch, seq_len, H)
        pred = self.fc(outputs)                                 # (batch, seq_len, 1)
        return pred, (H, C)

In [4]:
# Define the LSTM Model
class LSTMModel(nn.Module):
    def __init__(self):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=50, num_layers=1, batch_first=True)
        self.fc = nn.Linear(50, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])  # Use the last output of the LSTM
        return out

In [ ]:
# Check the hand-written cell against nn.LSTM: copy the weights over and compare.
# nn.LSTM uses the same gate order (i, f, g, o) but has two biases, b_ih + b_hh,
# so put ours in b_ih and zero b_hh.
torch.manual_seed(0)
custom = CustomLSTMModel(input_dim=3, hidden_units=8)
reference = nn.LSTM(input_size=3, hidden_size=8, batch_first=True)
with torch.no_grad():
    reference.weight_ih_l0.copy_(custom.W_ih)
    reference.weight_hh_l0.copy_(custom.W_hh)
    reference.bias_ih_l0.copy_(custom.bias)
    reference.bias_hh_l0.zero_()

x = torch.randn(4, 7, 3)
H0, C0 = torch.randn(4, 8), torch.randn(4, 8)
with torch.no_grad():
    pred, (H, C) = custom(x, (H0, C0))
    out, (H_ref, C_ref) = reference(x, (H0.unsqueeze(0), C0.unsqueeze(0)))  # nn.LSTM states are (num_layers, batch, hidden)
assert torch.allclose(pred, custom.fc(out), atol=1e-5)
assert torch.allclose(H, H_ref[0], atol=1e-5) and torch.allclose(C, C_ref[0], atol=1e-5)
print("CustomLSTMModel matches nn.LSTM")

In [5]:
# Initialize the model, loss function, and optimizer
model_custom = CustomLSTMModel(1, 50)
model_inbuilt = LSTMModel()
criterion = nn.MSELoss()
optimizer_custom = optim.Adam(model_custom.parameters(), lr=0.01)
optimizer_inbuilt = optim.Adam(model_inbuilt.parameters(), lr=0.01)

In [ ]:
# Training loop for the custom model
epochs = 500
for epoch in range(epochs):
    # Forward pass
    state = None
    pred, state = model_custom(X_seq, state)
    loss = criterion(pred[:, -1, :], y_seq) # Use the last output of the LSTM
    # Backward pass and optimization
    optimizer_custom.zero_grad()
    loss.backward()
    optimizer_custom.step()

    # Log progress every 50 epochs
    if (epoch + 1) % 50 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")

In [ ]:
# Training loop for the inbuilt model
epochs = 500
for epoch in range(epochs):
    # Forward pass
    pred = model_inbuilt(X_seq)
    loss = criterion(pred, y_seq)
    # Backward pass and optimization
    optimizer_inbuilt.zero_grad()
    loss.backward()
    optimizer_inbuilt.step()

    # Log progress every 50 epochs
    if (epoch + 1) % 50 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")

In [ ]:
# Testing on new data: continue the sine wave past the training range, with the same step size
test_steps = 100  # Ensure this is greater than sequence_length
dt = (X[1] - X[0]).item()
X_test = X[-1] + dt * torch.arange(1, test_steps + 1).unsqueeze(1)
y_test = torch.sin(X_test)

# Create test input sequences and their next-step targets
X_test_seq, y_test_next = create_in_out_sequences(y_test, sequence_length)

with torch.no_grad():
    pred_custom, _ = model_custom(X_test_seq)
    pred_inbuilt = model_inbuilt(X_test_seq)
pred_custom = torch.flatten(pred_custom[:, -1, :])
pred_inbuilt = pred_inbuilt.squeeze()
y_test_next = y_test_next.squeeze()
print(f"Test MSE - custom model: {torch.mean((pred_custom - y_test_next) ** 2):.2e}, "
      f"in-built model: {torch.mean((pred_inbuilt - y_test_next) ** 2):.2e}")

In [ ]:
#Plot the predictions
plt.figure()
plt.plot(y_test_next, label="Ground Truth")
plt.plot(pred_custom, label="custom model")
plt.plot(pred_inbuilt, label="inbuilt model")
plt.legend()
plt.show()